# FloodA5 notebook 1 — the planar test cases

This notebook documents, runs and analyses FloodA5's three **planar test cases**,
simple synthetic terrains on which we know what the right answer looks like:

| Case | Question it answers |
|---|---|
| `planar_slope` | On a perfectly uniform slope, does the plume from a point source stay symmetric, as the terrain demands — on the pentagonal A5 mesh as well as on a square (Cartesian) grid? |
| `planar_embankment` | Does the sub-grid (SGS) flow model let water through a gap in an embankment that is much narrower than a mesh cell, while the standard model, which only sees each cell's mean elevation, holds it back? |
| `planar_wall` | The reverse question: does a wall with **no** gap hold water back when it is narrower than a cell and runs through the inside of cells? |

For each case it describes what the case tests and why, how its terrain and
meshes are made (including the Python steps), the exact commands it runs,
the checks it must pass, and figures and numbers that show the result.
It is the first of the notebooks (`notebooks/README.md`): `02_channel.ipynb`
(the straight-channel series), `03_carlisle.ipynb` (the Carlisle 2005
flood) and `04_sgs_strip.ipynb` (SGS against analytic answers) follow.

**How to run it.** Open it in VS Code (or Jupyter) with the kernel
**"Julia FloodA5 1.12"** (one-time set-up: `notebooks/README.md`) and run all
cells. The first run makes the case runs — terrain, meshes and thirteen
simulations, about 30 minutes on a desktop — in `test/case_runs/` (not
tracked by git). Later runs find them there and only analyse and plot,
which takes a minute or two. Nothing in this notebook changes files in the
repository.

**Words used throughout.**
*DEM* (digital elevation model): a raster of ground elevations.
*Mesh*: the cells the model computes on; each cell has one water volume
and exchanges water with its neighbours across shared edges.
*A5*: the pentagonal discrete global grid FloodA5 is built on; its
*resolution* sets the cell size (each level divides a cell into four:
res 16 cells are about 90 m across, res 18 about 22 m).
*Cartesian*: an ordinary grid of square cells, used as a comparison.
*Standard* flow model: the local-inertial scheme (Bates et al. 2010) with
each cell represented by its mean ground elevation.
*SGS* (sub-grid sampling): the same scheme, but each cell keeps tables
built from the DEM inside it — how much water it holds and how much of it
is wet at each water level — and each face between two cells keeps a table
of the flow area and conveyance its own ground profile offers at each
level, counting only the parts of the face that the water of both cells can
reach. So it can see terrain smaller than a cell (`docs/HYDRAULICS.md` §9;
rewritten in 0.7.0).
*WSE*: water surface elevation (ground + depth). An SGS cell's depth is
measured from its **lowest** ground, a standard cell's from its mean ground,
so the two depths are not the same quantity; where the notebook compares
them it uses the *mean depth* V/A (stored volume over cell area), which
means the same in both.

## 0. Set-up

The cell below finds the repository from the Julia project the kernel was
started with (so the notebook works from any checkout, whatever the working
directory), and loads three modules:

- `FA` — `analysis/FloodA5Analysis.jl`: reads output files (`FA.load_run`) and
  computes metrics from them (depths, water levels, stored volume, plume shape);
- `FC` — `cases/FloodA5Cases.jl`: the **test-case registry**. Each case is
  defined once, in `cases/registry/<case>.jl`, with the steps that make its
  inputs, the simulations it runs, the metrics it computes and the checks
  they must pass. `FC.show_case` prints every command; `FC.run_case` makes
  whatever is missing, then analyses and checks;
- `FP` — `visualisation/FloodA5Plots.jl`: static figures (CairoMakie).

`RUN_CASES = false` makes the notebook only analyse runs that already exist.
`PARALLEL` and `THREADS` set how many simulations run at once and how many
threads each uses.

In [ ]:
# The kernel starts Julia with --project=<repository> (or <repository>/ci):
# walk up from that project to the directory holding FloodModel.jl.
const REPO = let d = dirname(something(Base.active_project(), joinpath(pwd(), "Project.toml")))
    while !isfile(joinpath(d, "FloodModel.jl"))
        dirname(d) == d && error("no FloodA5 repository above $(Base.active_project())")
        d = dirname(d)
    end
    d
end
ENV["FLOODA5_NO_VIS"] = "1"            # the runs need no viewer

include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));     const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));           const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));   const FP = FloodA5Plots
using CairoMakie, Printf, Markdown, Statistics
FP.activate!()                         # the FloodA5 figure style

const RUN_CASES = true                 # false: only analyse existing runs
const PARALLEL  = 2                    # simulations at once
const THREADS   = 3                    # threads per simulation

println("repository : ", REPO)
println("version    : ", strip(read(joinpath(REPO, "VERSION"), String)))
println("case runs  : ", FC.case_root())

Small helpers: `mdtable` shows a table in the notebook, `f` formats a
number, and `meandepth_run` / `mesh_run` let the map functions draw a mean
depth or a mesh property.

In [ ]:
"An HTML table from a header and rows of already formatted values. (A Markdown table also carries
a LaTeX copy, a `tabular` environment, which VS Code's KaTeX renderer cannot draw.)"
function mdtable(header, rows)
    esc(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    cell(tag, x) = "<$tag style=\"text-align:right\">" * esc(x) * "</$tag>"
    s = "<table><thead><tr>" * join(cell("th", h) for h in header) * "</tr></thead><tbody>"
    for r in rows
        s *= "<tr>" * join(cell("td", v) for v in r) * "</tr>"
    end
    return HTML(s * "</tbody></table>")
end
"`x` to `d` decimals, or as 1.2e-05 if it would round to zero."
f(x, d = 3) = !isfinite(x) ? string(x) :
              (x != 0 && abs(x) < 0.5 * 10.0^-d) ? @sprintf("%.1e", x) : string(round(x; digits = d))

"""
Mean depth V/A (m) of run `r` at frame `k` — the same quantity for SGS and standard
cells (an SGS depth is measured from the cell's lowest ground) — as a one-frame run
whose `elevation` holds it (NaN at or below `dry`), so that `FP.fieldmap!(…; var = :elevation)` draws it.
"""
function meandepth_run(r, k; dry = 0.01)
    V = FA.volume(r, k); g = r.mesh
    d = [isfinite(V[i]) && V[i] / g.area[i] > dry ? V[i] / g.area[i] : NaN for i in eachindex(V)]
    m = FA.MeshGeom(g.ids, g.lon, g.lat, d, g.area, g.polygons, g.neighbours, g.static, g.id_scheme)
    return FA.Run("", r.label, 2, [r.t[k]], m, Dict{String,Any}(), "")
end
"A mesh alone as a one-frame run, for FP's maps (its `elevation` is drawn with `var = :elevation`)."
mesh_run(m, label) = FA.Run("", label, 2, [0.0], m, Dict{String,Any}(), "")
nothing

## 1. How a test case is made

All three cases follow the same four steps. The registry runs each step only if
its output is missing, so nothing is made twice.

1. **Terrain** — a Python script writes the DEM (a GeoTIFF), a parameters
   file and the area of interest (AOI, the domain outline as GeoJSON).
2. **Meshes** — `FloodModel.jl --meshgen` covers the AOI with A5 cells and
   samples the DEM onto them (`--meshgen-cartesian` makes a square grid).
   A mesh is saved as a GeoParquet file and loaded by every run that uses it.
3. **Simulations** — one `julia FloodModel.jl …` process per run, each
   writing an HDF5 output file in `test/case_runs/<case>/runs/`. The output
   records the full command, the resolved options, the code version and the
   mesh, so a run is reused only if it was made with the same command, and any
   output can be re-run (`--rerun`) or analysed on its own.
4. **Analysis and checks** — the case's metrics are computed from the
   outputs (with `FA`), then tested: first *physical checks* (mass is
   conserved; the behaviour the case exists to show), then a comparison
   of every metric with the values recorded in `cases/expected/<case>.toml`,
   which catches any unintended change of the results.

### 1.1 The terrain generator (Python)

`test/planar_embankment/generate_planar_embankment_dem.py` builds a plane
sloping down from west to east, optionally with an embankment running
north–south across it and a gap (*notch*) through the embankment on the
centre line. With `--emb-height 0 --emb-width 0` it is a pure plane; with
`--notch-width 0` the embankment has no gap (a wall, `planar_wall`). Its
core, in `build_dem`, is a few lines of NumPy on a grid of pixel centres
`xg`, `yg` (metres from the south-west corner):

```python
z = z_west - slope * xg                                   # the plane; 0 m at the east edge
in_emb = (xg >= x_emb - half_w) & (xg <= x_emb + half_w)
z[in_emb] = z_at_emb + emb_height                         # flat-topped embankment (the crest)
in_notch = np.abs(yg - y_centre) <= 0.5 * notch_width
z[in_notch & in_emb] = z_at_emb                           # notch down to the natural ground (the sill)
```

It writes, into `--out`:
- `planar_embankment_dem.tif` — the DEM (geographic coordinates, EPSG:4326,
  centred at 0.017° W, 51° N), with rasterio if it is installed;
- `planar_embankment_params.json` — the key elevations (crest, sill, the
  elevation at the injection point), the geometry and the extent, which the
  cases use in their checks;
- `planar_embankment_aoi.geojson` — the domain rectangle, the AOI for mesh
  generation.

The injection point is always at 10% of the domain width from the west edge,
on the centre line. Because the embankment and the notch are the pixels whose
centres fall within them, their widths are whole numbers of pixels: the
embankment case uses 2 m pixels so that its 4 m notch is exactly 2 pixels.

**Where Python runs.** FloodA5 uses one Python: the one PyCall is built
with (Julia's Conda.jl environment, `~/.julia/conda/3`), into which
`setup.jl` installs pya5, NumPy, shapely, geopandas and pyarrow. On Windows
it needs conda's `Library\bin` on `PATH`, so the registry starts it from
Julia with that `PATH` (the long `julia -e 'using PyCall; …'` line that
`FC.show_case` prints below is the same command for a terminal).

### 1.2 Mesh generation (pya5)

`julia --project=. FloodModel.jl --meshgen AOI --meshres N --dem DEM --meshout MESH.parquet --mesh-only`
does, in order (`mesh/A5Grid.jl`, `mesh_for_aoi`):

1. **Cells (Python, pya5).** Julia writes the AOI to a temporary file and
   runs `mesh/a5_bridge.py mesh_for_aoi` with PyCall's Python. The bridge
   asks pya5 for every resolution-N cell that overlaps the AOI
   (`polygon_to_cells(…, {"containment": "overlapping"})`), then adds
   neighbours (`grid_disk`) whose pentagons intersect the AOI until none is
   left, so the domain is fully covered. For each cell it records the A5
   id, centre and boundary; two cells are neighbours if they share two
   boundary vertices (an edge); cell areas come from the boundary
   polygons. The result is written as GeoParquet, tagged with the pya5 ID
   scheme (`a5-v1` for pya5 1.x).
2. **Elevation (Julia).** The DEM is read with GDAL (ArchGDAL) and each
   cell's elevation is the **mean** of the DEM at 256 quasi-random (Halton)
   points inside it (`--dem-method mean`, the default).
3. **SGS tables (Julia, only with `--flow-model sgs`; `build_sgs_tables!`,
   `docs/HYDRAULICS.md` §9).** Each cell is covered with a regular
   *working grid* (about 1 600 points, never coarser than the DEM pixel) on
   which the DEM is sampled. From it come
   - the cell's **storage table**: stored volume and wetted area at 100
     water levels between its lowest and highest ground (`sgs_z_min`,
     `sgs_z_max`);
   - its **connectivity**: for every grid point, the lowest water level at
     which it is joined, through ground below that level, to the cell's
     *main basin* (the first low region that reaches two sides of the cell).

   Along every face the DEM is sampled every half pixel, and **one table per
   face**, held identically by its two cells, gives its flow area, wetted
   width and conveyance (Σ dx·h^(5/3) over the profile) at 64 levels from
   its sill up. A point of the face counts only once the water there is
   connected to the main basins of *both* cells, so a barrier inside a
   cell closes the faces behind it up to its crest (`sgs_face_raise`
   records by how much each face's sill was raised). The lowest level at
   which a face conveys is its *sill*.

Mesh generation runs on one thread (`--threads 1`): PyCall and GDAL are not
safe to use from several threads. `--meshgen-cartesian AOI --dx 25` instead
makes a grid of 25 m squares over the AOI's bounding box, in Julia alone.

The cell below runs a few lines of Python with that interpreter, in a
process of its own with conda's `PATH`, as FloodA5 itself does. It asks pya5
for the pentagon containing the planar-slope injection point at resolutions
16 and 18: its id, its five corners and its area. Mesh generation does the
same for every cell of the domain. (It needs pya5 installed; nothing later
depends on it.)

In [ ]:
py = FC.python_command()                    # PyCall's Python (FC asks a separate Julia process for it)
d = dirname(py)
script = join([
    "import sys, math, importlib.metadata as md",
    "import a5",
    "lon, lat = float(sys.argv[1]), float(sys.argv[2])",
    "print('pya5  :', md.version('pya5'))",
    "R = 6371000.0",
    "for res in (16, 18):",
    "    c = a5.lonlat_to_cell((lon, lat), res)",
    "    b = a5.cell_to_boundary(c)[:-1]      # the ring repeats its first corner",
    "    lat0 = sum(p[1] for p in b) / len(b)",
    "    xy = [(math.radians(p[0]) * R * math.cos(math.radians(lat0)), math.radians(p[1]) * R) for p in b]",
    "    area = abs(sum(xy[i][0] * xy[i-1][1] - xy[i-1][0] * xy[i][1] for i in range(len(xy)))) / 2",
    "    print(f'res {res}: cell {a5.u64_to_hex(c)}, {len(b)} corners, area {area:.0f} m2 (about {math.sqrt(area):.0f} m across)')",
], "\n")
lat_src, lon_src = FC.planar_point()  # the planar_slope injection point
println("Python: ", py)
env = FC.python_env([d, joinpath(d, "Library", "bin"), joinpath(d, "Scripts")])
print(read(setenv(`$py -c $script $lon_src $lat_src`, env), String))

## 2. `planar_slope` — is a plume on a uniform slope symmetric?

### What it tests and why

A pure plane, 4 km (west–east) × 2 km, sloping down to the east at 0.1%
(1 m per km), with water injected at a constant 0.1 m³/s at one point
400 m from the west edge on the centre line. Every boundary is closed, so
all the water stays in the domain. Nothing in the terrain distinguishes north
from south, so the exact solution is mirror-symmetric about the line through
the source: as much water north of it as south, and flow straight down the
slope (due east). Any asymmetry is an error of the mesh or the numerical
scheme — a *directional bias*.

This was FloodA5's main directional-bias benchmark. On A5 meshes the
pentagons are not arranged symmetrically about any line, and the original
scheme turned the plume off the slope by 14–15° (north/south asymmetry of
0.68 at res 16 and 0.87 at res 18; `docs/HYDRAULICS.md` §15.1). The 0.3.0
defaults — a face-gradient reconstruction that is consistent on pentagons
(*diamond*) and friction computed from the full velocity vector — reduced that
by an order of magnitude. A Cartesian grid aligned with the slope is
mirror-symmetric about the source row, so it should be symmetric to
round-off: a check that the measurement itself is unbiased.

The metrics, all about the source cell (`FA.planar_metrics`), at 5, 10, 15
and 20 h:

- **asym** = (V_N − V_S) / (V_N + V_S), V_N and V_S the water volumes north
  and south of the source, leaving out a ±37.5 m band along the centre line
  (cells there straddle the symmetry line). 0 for a symmetric plume, ±1 if
  all the water is on one side.
- **deflection**: the direction of the plume's centre of mass from the
  source, in degrees north of east. 0° is straight downslope.
- **A2**: a measure of lateral skew of the depth field (its second sine
  harmonic about the source); 0 for a symmetric plume.
- **flow direction** at 20 h: the speed-weighted mean direction of the
  velocity in the moving cells, and the fraction within ±30° of east.

### The commands

`FC.show_case` prints the case's description and every command it runs,
taken from the registry (so they cannot drift from what is actually run;
`{case}` paths are filled in for this checkout).

In [ ]:
ps = FC.case("planar_slope")
FC.show_case(ps)

Reading the commands:

- **prep dem** runs the generator with `--emb-height 0 --emb-width 0` (no
  embankment) and its defaults otherwise: 4 × 2 km, 0.1% slope, 5 m pixels.
- **prep a5_res16 / a5_res18** generate the two A5 meshes from the AOI and
  DEM (`--flow-model standard` means no SGS tables are built);
  **prep cart_dx25** the 25 m Cartesian grid.
- **run …** are the three simulations. `--meshload` loads a mesh;
  `--flow-model standard` selects the standard local-inertial model with its
  default scheme; `--injection-point lat,lon,Q` adds Q m³/s at the cell
  containing that point; `--closed-boundaries` makes every boundary a wall;
  `--sim-duration 72000` runs 20 h; `--dt-max 2` caps the time step at 2 s
  (as in the historical runs); `--output-interval 1800` writes a frame every
  30 minutes to `--output`. Manning's n is the default 0.03.

### Run and check

`FC.run_case` makes whatever is missing (here, on a first run, the DEM, three
meshes and three simulations, ~10 minutes), then computes the metrics and runs
the checks. Each check prints PASS or FAIL with the values behind it; the last
line compares every metric with the recorded expected values.

In [ ]:
t0 = time()
ok_ps = RUN_CASES ? FC.run_case(ps; parallel = PARALLEL, threads = THREADS) : FC.check(ps)
@printf("\nplanar_slope: %s (%.0f s)\n", ok_ps ? "all checks pass" : "SOME CHECKS FAIL", time() - t0)

### The domain and the meshes

The generator's parameters file gives the terrain; the output files carry the
meshes. The injection point the runs use is checked against the parameters
file by the first check above.

In [ ]:
psp = FC.planar_params(FC.case_dir(ps))          # the generator's parameters file
@printf("domain %.0f × %.0f m, %d × %d pixels of %.0f m; slope %.2f%%: z = %.1f m (west) to %.1f m (east)\n",
        psp["domain_m_ew"], psp["domain_m_ns"], psp["nx"], psp["ny"], psp["pixel_m"],
        psp["slope_pct"], psp["z_west"], psp["z_east"])
@printf("injection point: x = %.0f m, y = %.0f m (lon %.5f, lat %.5f), ground %.2f m\n",
        psp["injection_x_m"], psp["injection_y_m"], psp["injection_lon"], psp["injection_lat"], psp["injection_z"])

ps_label = Dict("a5_res16" => "A5 res 16", "a5_res18" => "A5 res 18", "cart_dx25" => "Cartesian 25 m")
ps_runs = [FA.load_run(FC.job_output(ps, j); label = ps_label[j.name]) for j in ps.jobs]
mdtable(["run", "cells", "mean cell size", "frames", "FloodA5"],
        [(r.label, length(r.mesh), @sprintf("%.1f m", sqrt(mean(r.mesh.area))), FA.nframes(r),
          get(r.info, "flooda5_version", "?")) for r in ps_runs])

### Results

The metrics the case computed (`FC.analyse` returns them as a dictionary per
run; they are also written to `test/case_runs/planar_slope/metrics.toml`):

In [ ]:
psm = FC.analyse(ps; io = devnull)
rows = [(ps_label[j], (f(psm[j]["asym_$(h)h"]) for h in (5, 10, 15, 20))...,
         f(psm[j]["deflection_20h"], 2) * "°", f(psm[j]["A2_20h"]),
         f(psm[j]["dir_mean_w"], 1) * "°", @sprintf("%.0f%%", 100 * psm[j]["dir_frac30"]),
         @sprintf("%.1f", psm[j]["storage_m3"]), @sprintf("%.0e", psm[j]["mass_balance_rel"]))
        for j in ("a5_res16", "a5_res18", "cart_dx25")]
mdtable(["run", "asym 5 h", "asym 10 h", "asym 15 h", "asym 20 h", "deflection 20 h", "A2 20 h",
         "mean flow direction", "within ±30° of east", "stored (m³)", "mass error"], rows)

**Depth maps.** Depth at 5, 10 and 20 h on the three meshes, in metres
from the source (★), zoomed on the plume. The water spreads as a very thin
sheet: about 1–2 cm deep at the source and well under a millimetre at the
plume edges, so cells are drawn wet from 0.1 mm (cells drier than that show the
grey terrain). The dashed lines mark the ±37.5 m band left out of the
north/south split. All panels share one colour scale.

In [ ]:
mc = FP.MapCoords(:local, psp["injection_lon"], psp["injection_lat"])   # metres from the source
ps_times = [5, 10, 20] .* 3600.0
K = FP.frame_grid(ps_runs, ps_times)
depths = [FA.depth(r, K[i, j]) for (i, r) in enumerate(ps_runs), j in eachindex(ps_times)]
const DRY_PS = 1e-4                    # m: wet from 0.1 mm
rng = FP.default_range(:depth, filter(>(DRY_PS), reduce(vcat, vec(depths))))

# zoom: the cells wet at 20 h in any run, plus a margin
xs = Float64[]; ys = Float64[]
for (i, r) in enumerate(ps_runs)
    x, y = FP.project(mc, r.mesh.lon, r.mesh.lat)
    w = findall(>(DRY_PS), depths[i, end]); append!(xs, x[w]); append!(ys, y[w])
end
ymax = maximum(abs, ys) + 60
xl = (-150.0, maximum(xs) + 100); yl = (-ymax, ymax)

PW = 400                               # panel width (px)
fig = Figure()
for (i, r) in enumerate(ps_runs), (j, t) in enumerate(ps_times)
    p = FP.fieldmap!(fig[i, j], r, K[i, j]; coords = mc, colorrange = rng, colorbar = false, dry_threshold = DRY_PS,
                     sources = false, title = j == 1 ? r.label : "", subtitle = FP.time_label(r.t[K[i, j]]))
    limits!(p.axis, xl..., yl...)
    p.axis.aspect = nothing; p.axis.width = PW; p.axis.height = PW * (yl[2] - yl[1]) / (xl[2] - xl[1])
    hlines!(p.axis, [-37.5, 37.5]; color = FP.INK2, linestyle = :dash, linewidth = 0.8)
    FP.mark_sources!(p, r; labels = false)
    i < length(ps_runs) && hidexdecorations!(p.axis; grid = false, ticks = false)
    j > 1 && hideydecorations!(p.axis; grid = false, ticks = false)
end
Colorbar(fig[1:length(ps_runs), length(ps_times) + 1]; colormap = cgrad(FP.SEQUENTIAL), limits = rng,
         label = "Depth (m)", height = Relative(0.6), width = 12)
resize_to_layout!(fig)
fig

**Asymmetry and deflection through time.** The same metrics at every output
frame (every 30 minutes). The grey dashed lines are the bounds the A5 runs
must stay within from 5 h on (|asym| < 0.2, |deflection| < 5°; the shaded
first 5 hours are not checked); before the 0.3.0
scheme the 20 h values were 0.68 / 15° (res 16) and 0.87 / 14° (res 18).
The right panel shows the direction of flow in the moving cells at 20 h
(0° = east, downslope; positive = north of east), weighted by speed.

In [ ]:
band = FC.PLANAR_SLOPE_BAND                                 # 37.5 m, as the case
src = (lon0 = psp["injection_lon"], lat0 = psp["injection_lat"])
styles = [(color = FP.SERIES[i], linestyle = :solid) for i in eachindex(ps_runs)]
fig = Figure(size = (1100, 380))
ax1 = Axis(fig[1, 1]; xlabel = "Time (h)", ylabel = "asym = (V_N − V_S)/(V_N + V_S)",
           title = "North/south volume asymmetry")
ax2 = Axis(fig[1, 2]; xlabel = "Time (h)", ylabel = "Deflection (° north of east)",
           title = "Plume deflection")
ax3 = Axis(fig[1, 3]; xlabel = "Flow direction (° north of east)", ylabel = "Share of speed-weighted flow",
           title = "Flow directions at 20 h")
for ax in (ax1, ax2)
    vspan!(ax, [0.0], [5.0]; color = (FP.GRID, 0.6))
end
hlines!(ax1, [-0.2, 0.2]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
hlines!(ax2, [-5.0, 5.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
for (i, r) in enumerate(ps_runs)
    ks = [k for k in 1:FA.nframes(r) if r.t[k] >= 1800 - 30]
    pm = [FA.planar_metrics(r, k; src..., band) for k in ks]
    lines!(ax1, r.t[ks] ./ 3600, getfield.(pm, :asym); color = styles[i].color, linewidth = 2, label = r.label)
    lines!(ax2, r.t[ks] ./ 3600, getfield.(pm, :deflection); color = styles[i].color, linewidth = 2)
    fd = FA.flow_direction_metrics(r, FA.nframes(r))
    edges = -45:5:45
    wsum = [sum(fd.speeds[(fd.angles .>= a) .& (fd.angles .< a + 5)]; init = 0.0) for a in edges[1:end-1]]
    stairs!(ax3, collect(edges), vcat(wsum ./ sum(fd.speeds), 0.0); step = :post,
            color = styles[i].color, linewidth = 2)
end
xlims!(ax3, -45, 45)
Legend(fig[2, 1:3], ax1; orientation = :horizontal, framevisible = false)
fig

### Interpretation

- **Mass.** All three runs store exactly the 7 200 m³ injected (0.1 m³/s ×
  72 000 s) and conserve mass to ~6·10⁻¹³ of it, i.e. to round-off.
- **Cartesian 25 m** is symmetric: |asym| ≈ 1·10⁻⁵ and deflection ≈ 5·10⁻⁵°
  at every time. The residual is not the scheme: a grid of constant
  longitude spacing is about 6·10⁻⁵ narrower 300 m north of the source than
  300 m south (the meridians converge). This confirms that the metric itself is
  unbiased.
- **A5** has a residual bias, because its pentagons are not arranged
  symmetrically about the source. At 20 h asym ≈ +0.089 at res 16 and +0.088
  at res 18 (about 9% more water north than south), and the plume's centre of
  mass lies 1.9° and 0.8° north of east. Over the checked period, 5–20 h,
  |asym| ≤ 0.09 and the deflection ≤ 2.9° on both meshes, against 0.68–0.87
  and 14–15° before the 0.3.0 scheme. In the first hours, while the plume
  covers only a few dozen cells, single cells decide the split: at res 18 the
  plume first leans south (asym −0.50 and deflection −17° at 30 minutes,
  −0.21 and −3.7° at 2 h, −0.08 and −0.6° at 5 h) and straightens as it
  grows, its asymmetry crossing zero at about 10 h; at res 16 asym stays
  within ±0.17 and the deflection within 3.3° throughout.
- **Flow directions** are downslope: every moving cell at res 16, and 96% at
  res 18 and on the Cartesian grid, flows within ±30° of east, and the
  speed-weighted mean direction is within 3.4° of east. On the coarse res-16
  mesh the flow passes from cell to cell along two directions, roughly 15°
  either side of east, which together point downslope; at res 18 and on the
  Cartesian grid it is concentrated around east.

So on a uniform slope the A5 meshes now keep a point-source plume close to
symmetric (≈ 9% more volume on one side and < 3° of deflection once the plume
spans many cells), with no sign of the systematic turning of the earlier
scheme, and the Cartesian grid gives the symmetric reference.

## 3. `planar_embankment` — water through a notch narrower than a cell

### What it tests and why

Real floodplains have features narrower than any affordable mesh cell:
embankments, walls, ditches, and gaps through them. The **standard** flow
model gives each cell one elevation, the mean of the DEM inside it, so a
narrow gap through an embankment only lowers the mean of the cells it
crosses a little, and the model behaves as if the embankment were (almost)
unbroken. **SGS** keeps, for each face between two cells, the flow area and
conveyance of the face's own ground profile, and its connectivity search
finds the gap as the lowest route through the cell (§1.2), so water can pass
the gap as soon as it reaches the gap's floor — and only there.

The terrain is a plane 2 km × 0.5 km sloping east at 0.5% (10 m at the west
edge, 0 m at the east), on a 2 m DEM. Across it, at x = 1000 m, is a
flat-topped embankment 40 m wide and 2 m high (crest 7.0 m), cut by a notch
4 m wide on the centre line down to the natural ground (the sill, 5.0 m).
A res-18 A5 cell is about 22 m across, so the notch is about a fifth of a
cell. 10 m³/s enters at x = 200 m on the centre line (ramped up over the first
10 minutes) for 6 h; every boundary is closed.

What should happen:
- **standard**: no water passes the embankment until the water upstream
  rises to the lowest *cell-mean* crest, about 6.7 m (the notch lowers the
  means of the cells it crosses by ~0.3 m); then it overtops.
- **SGS**: water passes through the notch once it reaches the notch, at
  about the sill level (5 m), ~2 m below the crest, so from early on there is
  water downstream; nowhere else does water cross below the crest.

Two quantities per output frame (every 2 minutes) measure this:
the **upstream WSE**, the highest water surface among the wet cells along
the embankment's upstream foot, and the **downstream volume**, the water
stored in cells entirely east of the embankment. Water has *passed* when the
downstream volume first exceeds 1 m³.

### The commands

In [ ]:
pe = FC.case("planar_embankment")
FC.show_case(pe)

- **prep dem**: the same generator, now with a 2 × 0.5 km domain, 2 m
  pixels, a 0.5% slope, and the embankment (`--emb-height 2 --emb-width 40
  --notch-width 4`).
- **prep a5_res18_sgs / a5_res18_standard**: the same res-18 A5 cells
  twice, once with SGS tables (`--flow-model sgs`, which builds them) and
  once without.
- **prep hydrograph**: writes the inflow hydrograph, a two-column CSV of
  time (s) and discharge (m³/s): 0 at t = 0, 10 m³/s from 600 s on.
- **run sgs / standard**: `--inflow-point lat,lon,FILE` adds the hydrograph's
  discharge at that point; the rest as in `planar_slope`, with 6 h of
  simulation and a frame every 120 s.

### Run and check

On a first run this makes the DEM, the two meshes (~1.5 min each) and the
two simulations (~2 min each). The case's expected values were re-recorded
for the 0.7.0 SGS scheme; those of the scheme up to 0.6.0 are kept in
`cases/expected/v0.6.0/planar_embankment_metrics.toml` and are shown
alongside below.

In [ ]:
t0 = time()
ok_pe = RUN_CASES ? FC.run_case(pe; parallel = PARALLEL, threads = THREADS) : FC.check(pe)
@printf("\nplanar_embankment: %s (%.0f s)\n", ok_pe ? "all checks pass" : "SOME CHECKS FAIL", time() - t0)

### The terrain and what each mesh sees

The parameters file gives the embankment geometry. The maps below zoom on the
notch: on the left the cell elevations the standard model uses (cell means),
on the right the lowest ground in each cell of the SGS mesh (`sgs_z_min`).
The black lines outline the embankment and the notch on the DEM, the white
lines the cells.

In [ ]:
pep = FC.planar_params(FC.case_dir(pe))
for (k, what) in [("domain_m_ew", "domain west–east (m)"), ("domain_m_ns", "domain south–north (m)"),
                  ("pixel_m", "DEM pixel (m)"), ("slope_pct", "slope (%)"), ("z_west", "ground at the west edge (m)"),
                  ("emb_x_m", "embankment centre line, x (m)"), ("emb_width_m", "embankment width (m)"),
                  ("emb_crest_z", "crest (m)"), ("notch_width_m", "notch width (m)"),
                  ("notch_sill_z", "notch sill (m)"), ("injection_x_m", "inflow point, x (m)"),
                  ("injection_z", "ground at the inflow point (m)")]
    @printf("  %-34s %8.2f\n", what, pep[k])
end

pe_label = Dict("sgs" => "SGS", "standard" => "standard")
pe_runs = Dict(j.name => FA.load_run(FC.job_output(pe, j); label = pe_label[j.name]) for j in pe.jobs)
sgs, std = pe_runs["sgs"], pe_runs["standard"]
@printf("\nmesh: %d res-18 cells, mean cell size %.1f m (both runs)\n", length(sgs.mesh), sqrt(mean(sgs.mesh.area)))

In [ ]:
# generator convention: x, y in metres from the south-west corner, 111 320 m per degree
m_lon = 111320.0 * cosd(pep["centre_lat"])
lonlat(x, y) = (pep["lon0"] + x / m_lon, pep["lat0"] + y / 111320.0)
xe0, xe1 = pep["emb_x_m"] .+ (-0.5, 0.5) .* pep["emb_width_m"]
yn0, yn1 = pep["notch_y_centre_m"] .+ (-0.5, 0.5) .* pep["notch_width_m"]
mcn = FP.MapCoords(:local, lonlat(pep["emb_x_m"], pep["notch_y_centre_m"])...)   # metres from the notch
outline(xa, xb, ya, yb) = FP.project(mcn, [lonlat(x, y)[1] for (x, y) in ((xa, ya), (xb, ya), (xb, yb), (xa, yb), (xa, ya))],
                                          [lonlat(x, y)[2] for (x, y) in ((xa, ya), (xb, ya), (xb, yb), (xa, yb), (xa, ya))])
# a view of the SGS run whose "elevation" is each cell's minimum ground
m = sgs.mesh
sgs_zmin = mesh_run(FA.MeshGeom(m.ids, m.lon, m.lat, m.static["sgs_z_min"], m.area, m.polygons, m.neighbours,
                                m.static, m.id_scheme), sgs.label)

zl = (-90.0, 90.0); zy = (-60.0, 60.0)
fig = Figure()
for (j, (r, title)) in enumerate([(std, "Standard mesh: cell-mean elevation"),
                                  (sgs_zmin, "SGS mesh: lowest ground in each cell")])
    p = FP.fieldmap!(fig[1, j], r, 1; var = :elevation, coords = mcn, colorrange = (4.6, 7.2),
                     colormap = :viridis, colorbar = false, sources = false, title, subtitle = "")
    limits!(p.axis, zl..., zy...)
    p.axis.aspect = nothing; p.axis.width = 480; p.axis.height = 480 * (zy[2] - zy[1]) / (zl[2] - zl[1])
    for P in FP.project_polygons(mcn, r.mesh)
        any(zl[1] - 30 .< P[1, :] .< zl[2] + 30) && any(zy[1] - 30 .< P[2, :] .< zy[2] + 30) &&
            lines!(p.axis, P[1, :], P[2, :]; color = (:white, 0.7), linewidth = 0.6)
    end
    lines!(p.axis, outline(xe0, xe1, 0.0, yn0)...; color = :black, linewidth = 1.2)
    lines!(p.axis, outline(xe0, xe1, yn1, pep["domain_m_ns"])...; color = :black, linewidth = 1.2)
    j > 1 && hideydecorations!(p.axis; grid = false, ticks = false)
end
Colorbar(fig[1, 3]; colormap = :viridis, limits = (4.6, 7.2), label = "Elevation (m)", height = Relative(0.7), width = 12)
resize_to_layout!(fig)
fig

In the standard mesh the embankment is a continuous ridge of cells: the
4 m notch only lowers the means of the cells it crosses. In the SGS mesh the
cells on the notch have their lowest ground at the natural surface, an open
route through the embankment. (What SGS conveys through are the faces'
sills, set by the connectivity search: the notch is the lowest route through
the cells it crosses, so their faces open at the notch's floor, while the
faces of cells cut off by the embankment stay closed up to its crest.) The
case measures the open route as a property of the meshes, *the lowest level
at which upstream cells connect to downstream cells through neighbouring
cells*:

In [ ]:
pem = FC.analyse(pe; io = devnull)
@printf("standard, through cell-mean elevations : %.3f m  (crest %.1f m)\n", pem["standard"]["z_bneck_mean"], pem["standard"]["crest_z"])
@printf("SGS, through each cell's lowest ground  : %.3f m  (sill %.1f m; ground at the embankment's upstream foot %.2f m)\n",
        pem["sgs"]["z_bneck_zmin"], pem["sgs"]["sill_z"], pem["sgs"]["z_foot"])

### Results

The upstream WSE and the downstream volume through time, computed exactly as
the case computes them (the cell below checks that they match the case's
hourly values):

In [ ]:
"Per frame: the upstream WSE (highest WSE of the wet cells along the embankment's
upstream foot) and the volume stored east of the embankment — as the case's analysis."
function pe_series(r, p)
    xmin, xmax = FC.embankment_xrange(r.mesh, p)
    xw = p["emb_x_m"] - p["emb_width_m"] / 2; xe = p["emb_x_m"] + p["emb_width_m"] / 2
    cell = sqrt(sum(r.mesh.area) / length(r.mesh.area))
    up = xmax .<= xw; down = xmin .>= xe
    foot = findall(up .& (xmax .>= xw - cell))                 # upstream cells along the foot
    n = FA.nframes(r); wse_up = fill(NaN, n); v_down = zeros(n)
    for k in 1:n
        h = FA.depth(r, k); η = FA.wse(r, k); V = FA.volume(r, k)
        w = [i for i in foot if isfinite(h[i]) && h[i] > FC.EMBANKMENT_WET_DEPTH]
        isempty(w) || (wse_up[k] = maximum(η[w]))
        v_down[k] = sum(V[i] for i in findall(down) if isfinite(V[i]); init = 0.0)
    end
    return (; t = r.t, wse_up, v_down)
end
pe_s = Dict(name => pe_series(r, pep) for (name, r) in pe_runs)
for name in ("sgs", "standard"), h in 1:6
    k = FA.frame_at(pe_runs[name], 3600.0 * h; tol = 90.0)      # as the case
    @assert pe_s[name].wse_up[k] ≈ pem[name]["wse_up_$(h)h"] && pe_s[name].v_down[k] ≈ pem[name]["v_down_$(h)h"]
end

hm(t) = (mins = round(Int, t / 60); @sprintf("%d:%02d", mins ÷ 60, mins % 60))
# the scheme up to 0.6.0, as recorded before the 0.7.0 rewrite of SGS (standard flow is unchanged)
pe060 = FC.TOML.parsefile(joinpath(REPO, "cases", "expected", "v0.6.0", "planar_embankment_metrics.toml"))
pe_rows = Dict("sgs" => pem["sgs"], "standard" => pem["standard"], "sgs060" => pe060["sgs"])
pe_rowlabel = Dict("sgs" => "SGS", "standard" => "standard", "sgs060" => "SGS, 0.6.0 scheme")
mdtable(["run", "water first passes", "upstream WSE then", "upstream WSE at 6 h", "downstream at 6 h",
         "stored at 6 h", "mass error"],
        [(pe_rowlabel[j], @sprintf("%.0f s (%s)", d["t_first_down"], hm(d["t_first_down"])),
          @sprintf("%.2f m", d["wse_up_at_first_down"]), @sprintf("%.2f m", d["wse_up_6h"]),
          @sprintf("%.0f m³", d["v_down_6h"]), @sprintf("%.0f m³", d["stored_m3"]),
          @sprintf("%.0e", d["mass_balance_rel"])) for (j, d) in ((j, pe_rows[j]) for j in ("sgs", "standard", "sgs060"))])

In [ ]:
mdtable(["time", "upstream WSE, SGS", "standard", "SGS 0.6.0", "downstream, SGS", "standard", "SGS 0.6.0"],
        [("$h h", (@sprintf("%.2f m", pe_rows[j]["wse_up_$(h)h"]) for j in ("sgs", "standard", "sgs060"))...,
          (@sprintf("%.0f m³", pe_rows[j]["v_down_$(h)h"]) for j in ("sgs", "standard", "sgs060"))...)
         for h in 1:6])

**Through time.** Left: the upstream WSE against the sill, the standard
mesh's cell-mean crest and the true crest. Right: the volume downstream of
the embankment; the dotted verticals mark when water first passes in each run.

In [ ]:
pe_col = Dict("sgs" => FP.SERIES[1], "standard" => FP.SERIES[2])
fig = Figure(size = (1100, 420))
ax1 = Axis(fig[1, 1]; xlabel = "Time (h)", ylabel = "Upstream WSE (m)", title = "Water level upstream of the embankment")
ax2 = Axis(fig[1, 2]; xlabel = "Time (h)", ylabel = "Downstream volume (10³ m³)", title = "Water that has passed the embankment")
for (z, lab) in [(pep["notch_sill_z"], "sill"), (pem["standard"]["z_bneck_mean"], "cell-mean crest (standard)"),
                 (pep["emb_crest_z"], "crest")]
    hlines!(ax1, [z]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
    text!(ax1, 0.1, z; text = lab, color = FP.INK2, fontsize = 11, align = (:left, :bottom))
end
for name in ("sgs", "standard")
    s = pe_s[name]
    lines!(ax1, s.t ./ 3600, s.wse_up; color = pe_col[name], linewidth = 2, label = pe_label[name])
    lines!(ax2, s.t ./ 3600, s.v_down ./ 1e3; color = pe_col[name], linewidth = 2)
    vlines!(ax2, [pem[name]["t_first_down"] / 3600]; color = pe_col[name], linestyle = :dot, linewidth = 1.5)
end
ylims!(ax1, 4.8, 7.3)
Legend(fig[2, 1:2], ax1; orientation = :horizontal, framevisible = false)
fig

**Maps.** Mean depth (V/A, the same quantity in both models) on the two
meshes when SGS first passes water through the notch, when the standard
model first overtops, and at the end of the run; cells with less than 1 cm
are not drawn. The dashed lines mark the embankment; the star is the inflow
point.

In [ ]:
mce = FP.MapCoords(:local, lonlat(0.0, pep["notch_y_centre_m"])...)        # x from the west edge, y from the centre line
pe_times = [pem["sgs"]["t_first_down"], pem["standard"]["t_first_down"], 6 * 3600.0]
runs2 = [sgs, std]
Ke = FP.frame_grid(runs2, pe_times)
mdr = [meandepth_run(r, Ke[j, i]) for i in eachindex(pe_times), (j, r) in enumerate(runs2)]
rng = FP.default_range(:depth, filter(isfinite, reduce(vcat, [x.mesh.elevation for x in vec(mdr)])))
xa = FP.project(mce, [lonlat(x, 0.0)[1] for x in (xe0, xe1)], [pep["lat0"], pep["lat0"]])[1]
fig = Figure()
for (i, t) in enumerate(pe_times), (j, r) in enumerate(runs2)
    p = FP.fieldmap!(fig[i, j], mdr[i, j], 1; var = :elevation, colormap = cgrad(FP.SEQUENTIAL), basemap = :grey,
                     coords = mce, colorrange = rng, colorbar = false, sources = false,
                     title = i == 1 ? r.label : "", subtitle = FP.time_label(r.t[Ke[j, i]]), width = 520)
    vlines!(p.axis, collect(xa); color = FP.INK2, linestyle = :dash, linewidth = 0.8)
    FP.mark_sources!(p, r; labels = false)
    i < length(pe_times) && hidexdecorations!(p.axis; grid = false, ticks = false)
    j > 1 && hideydecorations!(p.axis; grid = false, ticks = false)
end
Colorbar(fig[1:3, 3]; colormap = cgrad(FP.SEQUENTIAL), limits = rng, label = "Mean depth V/A (m)", height = Relative(0.6), width = 12)
resize_to_layout!(fig)
fig

**Water balance.** The domain is closed, so the stored volume must equal
the volume of the hydrograph: 10 m³/s × (21 600 s − 300 s) = 213 000 m³ (the
ramp over the first 600 s counts half). Left: storage in both runs; right:
the SGS run's mass balance — stored volume through time, the summary's
added − removed volume at the end, and the net inflow rate between frames.

In [ ]:
fig = Figure(size = (1150, 460))
FP.storage_plot!(fig[1, 1], [sgs, std])
FP.mass_balance_plot!(fig[1, 2], sgs)
fig

**Earlier observations, checked again.** Under the 0.6.0 scheme this
notebook recorded that the SGS wetting front reached the embankment in about
12 minutes against 35 for standard, that SGS stood much deeper at the inflow
point, that the downstream water leaned north, and that the notch conveyed
only about 3 m³/s. The cell below measures each of them on the current runs:

- *front at the embankment*: the first frame at which a cell along the
  embankment's upstream foot holds more than 1 cm — by mean depth V/A (the
  like-for-like measure) and, for SGS, also by its own depth above the cell's
  lowest ground;
- *depth at the inflow*: the cell the inflow enters, SGS depth (above its
  lowest ground) and mean depth, against standard;
- *lean*: the share of the downstream volume north of the notch's centre line;
- *notch discharge*: the growth rate of the downstream volume, with the
  upstream level, late in the run.

In [ ]:
"First time (s) at which any of `cells` holds more than `thr` (m) of `what` (:mean = V/A, :depth)."
function first_wet(r, cells, thr; what = :mean)
    for k in 1:FA.nframes(r)
        v = what === :mean ? FA.volume(r, k) ./ r.mesh.area : FA.depth(r, k)
        any(i -> isfinite(v[i]) && v[i] > thr, cells) && return r.t[k]
    end
    return Inf
end
pe_obs = Dict{String,Any}()
for (name, r) in pe_runs
    xmin, xmax = FC.embankment_xrange(r.mesh, pep)
    cell = sqrt(mean(r.mesh.area))
    foot = findall((xmax .<= xe0) .& (xmax .>= xe0 - cell)); down = findall(xmin .>= xe1)
    src = FA.source_cell(r.mesh, pep["injection_lon"], pep["injection_lat"])
    yc = (r.mesh.lat .- pep["lat0"]) .* 111320.0                       # cell centre, m from the south edge
    north(k) = (V = FA.volume(r, k); vn = sum(V[i] for i in down if yc[i] > pep["notch_y_centre_m"]; init = 0.0);
                vn / max(sum(V[down]), eps()))
    kh(h) = FA.frame_at(r, 3600.0 * h; tol = 90.0)
    o = (front_mean = first_wet(r, foot, 0.01), front_depth = first_wet(r, foot, 0.01; what = :depth),
         src_depth = [FA.depth(r, kh(h))[src] for h in (1, 6)],
         src_mean = [FA.volume(r, kh(h))[src] / r.mesh.area[src] for h in (1, 6)],
         north = [north(kh(h)) for h in (1, 2, 6)],
         q_down = (pe_s[name].v_down[kh(6)] - pe_s[name].v_down[kh(5)]) / (r.t[kh(6)] - r.t[kh(5)]),
         wse5 = pe_s[name].wse_up[kh(5)], wse6 = pe_s[name].wse_up[kh(6)])
    pe_obs[name] = o
    @printf("%-8s front at the foot: mean depth > 1 cm at %s, own depth > 1 cm at %s\n", pe_label[name],
            hm(o.front_mean), hm(o.front_depth))
    @printf("         inflow cell at 1 h / 6 h: depth %.2f / %.2f m, mean depth %.2f / %.2f m\n",
            o.src_depth..., o.src_mean...)
    @printf("         downstream volume north of the notch line: %.0f%% (1 h), %.0f%% (2 h), %.0f%% (6 h)\n", (100 .* o.north)...)
    @printf("         downstream volume grows by %.1f m³/s over 5–6 h (inflow 10 m³/s); upstream WSE %.3f → %.3f m\n",
            o.q_down, o.wse5, o.wse6)
end
@printf("\n0.6.0 scheme (recorded): SGS downstream volume grew by %.1f m³/s over 5–6 h; upstream WSE %.3f → %.3f m\n",
        (pe060["sgs"]["v_down_6h"] - pe060["sgs"]["v_down_5h"]) / 3600, pe060["sgs"]["wse_up_5h"], pe060["sgs"]["wse_up_6h"])

### Interpretation

- **Standard holds the water back until it reaches the crest it sees.**
  Its lowest route across the embankment through cell-mean elevations is
  6.72 m, 0.28 m below the true crest: the notch is averaged away. No water
  passes for 4.5 h (16 202 s), while the upstream level rises from the
  natural ground to 6.77 m; only then does it overtop, and by 6 h
  12 600 m³ is downstream.
- **SGS passes water through the notch at the sill.** Its lowest route is
  5.11 m, the ground at the embankment's upstream foot (5.1 m) — the notch
  floor (5.0 m) is lower still, so the notch is open. Water passes after
  about 14 minutes (842 s; 18 minutes, 1 082 s, under the 0.6.0 scheme), as
  soon as the plume reaches the embankment, with the upstream level at
  5.25 m, 1.75 m below the crest.
- **It passes only at the notch.** The case's crossing metrics (`cross_*`,
  `n_row_far_wet_*`, `v_emb_*`) find no water on the embankment's top away
  from the notch and no cell of the first downstream row more than 100 m
  from it wet at any hour; water downstream away from the notch has spread
  along the downstream toe from the notch.
- **The notch now carries nearly all the inflow.** Late in the run the
  downstream volume grows by almost 10 m³/s and the upstream level has nearly
  stopped rising (6.07 m at 6 h, about 1.1 m above the sill): the notch passes
  what comes in, and by 6 h about three quarters of the inflow is downstream.
  Under the 0.6.0 scheme it carried about 3 m³/s at a higher head and the pond
  kept rising (6.22 m at 6 h, 39 000 m³ downstream): its face flow areas
  stopped growing at the cell's highest DEM sample (§2c of `TEST_CASES.md`,
  notebook 4).
- **The wetting front moves faster with SGS, by any measure.** By mean depth
  (more than 1 cm somewhere along the embankment's foot) it arrives after
  about 14 minutes with SGS and 34 with standard; by SGS's own depth, earlier
  still (8 minutes). (The 0.6.0 notebook recorded about 12 and 35 minutes.) A
  thin sheet flows faster when it is gathered in the lowest part of each cell
  than when it is spread over the cell; on this slope each 22 m cell spans
  about 0.1 m of elevation. The decisive difference is still the level at
  which water passes: 5.25 m against 6.77 m.
- **SGS is not deeper at the inflow any more.** The inflow cell stands
  0.19 m above its lowest ground (mean depth 0.14 m) against 0.16 m in
  standard flow, steady from the first hour.
- **The early downstream water leans north, then evens out.** At 1 h most of
  the (small) downstream volume of SGS lies north of the notch's centre line;
  by 6 h the split is nearly even (the standard run, which passes water only
  at the end, is even). The early lean is where the first water spreads along
  the toe on this A5 mesh, not a lasting bias.
- **The difference is large while it matters.** At every output frame
  between SGS's first passing and standard's, SGS has more water downstream;
  just before standard overtops it is about 100 000 m³ against a few m³.
- **Mass.** Both runs store 212 990 m³, conserving mass to ~2·10⁻¹⁵. The
  10 m³ (5·10⁻⁵) below the hydrograph's 213 000 m³ is the first-order time
  integration of the inflow ramp (`input_error_rel` in the metrics), not a
  loss.

So the SGS tables let a sub-cell gap act as a gap: at a fifth of a cell wide
it carries water from the moment water reaches it, and nowhere else, while
the standard model treats the embankment as unbroken until the water reaches
its cell-mean crest. `planar_wall` (§4) asks the reverse: whether SGS also
treats a sub-cell wall *without* a gap as a wall.

## 4. `planar_wall` — is a wall inside a cell a wall?

### What it tests and why

`planar_embankment` asks whether SGS sees a *gap* narrower than a cell. This
case asks the reverse: does SGS see a *barrier* narrower than a cell, one
that runs through the inside of cells rather than along their faces?

SGS stores each cell's water as one level pool: all the ground in the cell
below the water level holds water, whether or not it is connected. Until
0.6.0 a face also conveyed according to the ground profile on the face line
alone. So a wall that crossed the inside of a cell was not a barrier: the
faces on both sides of it had natural ground, water entered over the
upstream faces, was stored at the cell's lowest ground (on a slope, behind
the wall) and left through the downstream faces. The wall only reduced the
cell's stored volume. The case was added to measure this (SGS fix plan,
Phase 0), and it did: **until 0.6.0 SGS passed 58–64% of the inflow past a
4 m wall, from the moment the water reached it, 2.9 m below its crest**
(`docs/TEST_CASES.md` §2d; `CHANGELOG.md`).

The terrain is the `planar_embankment` plane (2 × 0.5 km, 0.5% west→east,
2 m DEM) with a wall 3 m high across it and **no notch**, in two variants,
each with its own DEM and meshes:

- `wide`: 40 m, x = 980–1020 m, crest 8.00 m — wider than a cell on both
  meshes, so some cells lie wholly on it;
- `thin`: 4 m (2 pixels), x = 988–992 m, crest 8.05 m — narrower than any
  cell, and placed inside the cells of both meshes (on the Cartesian grid,
  inside the column x = 981.1–1001.1 m, at least 7 m from its faces).

10 m³/s enters at x = 200 m on the centre line (10-minute ramp) for 6 h,
every boundary closed: 213 000 m³. Held as a level pool against the wall's
upstream face that volume is a wedge, V(η) = ½(η − z_w)²·W/S =
5·10⁴ (η − z_w)² m³ (S = 0.005, W = 500 m, z_w the ground at the face), so
the water ends at most 2.06 m above the ground at the wall, about 0.9 m below
the crest. **In a correct model no water passes either wall.** Eight runs:
SGS and standard flow, on A5 res 18 and Cartesian 20 m, for each wall.

### The commands

In [ ]:
pw = FC.case("planar_wall")
FC.show_case(pw)

- **prep dem_wide / dem_thin**: the generator with `--emb-height 3`, the
  wall's width and position, and `--notch-width 0` (no gap), each into its
  own directory.
- **prep a5_… / cart_…**: per wall, the A5 res-18 and Cartesian 20 m meshes
  with SGS tables (`--flow-model sgs`, connectivity on by default) and
  without (`standard`).
- **run …**: as `planar_embankment`, with the mesh and model of each run.

(The case's description, written before 0.7.0, still calls the SGS
thin-wall checks expected failures; since 0.7.0 they are ordinary checks and
pass — the checks themselves, printed below, say so.)

### Run and check

On a first run this makes two DEMs, eight meshes (~10 minutes) and eight
simulations (~1 minute each).

In [ ]:
t0 = time()
ok_pw = RUN_CASES ? FC.run_case(pw; parallel = PARALLEL, threads = THREADS) : FC.check(pw)
@printf("\nplanar_wall: %s (%.0f s)\n", ok_pw ? "all checks pass" : "SOME CHECKS FAIL", time() - t0)

### The design and the result in numbers

The case's metrics (`FC.analyse`): per wall the design (crest, ground at the
wall's upstream face, the level-pool level of the whole inflow), and per run
what reached the downstream side.

In [ ]:
pwm = FC.analyse(pw; io = devnull)
WALLS = ("thin", "wide")
PW_MESH = Dict("a5" => "A5 res 18", "cart" => "Cartesian 20 m")
PW_MODEL = Dict("sgs" => "SGS", "std" => "standard")
pw_job(mesh, model, wall) = "$(mesh)_$(model)_$(wall)"
pw_label(j) = (p = split(j, '_'); "$(PW_MODEL[p[2]]), $(PW_MESH[p[1]])")
mdtable(["wall", "width", "crest", "ground at the wall's upstream face", "level-pool level of 213 000 m³", "freeboard"],
        [(w, @sprintf("%.0f m", pwm[pw_job("a5", "sgs", w)]["wall_width_m"]), @sprintf("%.2f m", pwm["design_$w"]["crest_z"]),
          @sprintf("%.2f m", pwm["design_$w"]["z_foot"]), @sprintf("%.2f m", pwm["design_$w"]["wse_end"]),
          @sprintf("%.2f m", pwm["design_$w"]["freeboard"])) for w in WALLS])

In [ ]:
pw_jobs = [pw_job(mesh, model, w) for w in WALLS for model in ("sgs", "std") for mesh in ("a5", "cart")]
fmt_t(t) = isfinite(t) ? @sprintf("%.0f s (%s)", t, hm(t)) : "never"
mdtable(["wall", "run", "downstream at 6 h", "share of the inflow", "first downstream water",
         "level at the wall then", "highest upstream level", "lowest cell-mean route"],
        [(last(split(j, '_')), pw_label(j), @sprintf("%.0f m³", pwm[j]["v_down_end"]),
          @sprintf("%.1f%%", 100 * pwm[j]["frac_down_end"]), fmt_t(pwm[j]["t_first_down"]),
          isfinite(pwm[j]["t_first_down"]) ? @sprintf("%.2f m", pwm[j]["wse_wall_at_first_down"]) : "–",
          @sprintf("%.2f m", pwm[j]["wse_up_max"]), @sprintf("%.2f m", pwm[j]["z_bneck_mean"])) for j in pw_jobs])

For comparison, the scheme up to 0.6.0 (`docs/TEST_CASES.md` §2d): SGS on the
thin wall passed 124 660 m³ (58.5% of the inflow) on A5, first at 722 s, and
136 972 m³ (64.3%) on the Cartesian grid, first at 962 s, with the water at
the wall about 5.1–5.2 m — 2.9 m below the crest — and the upstream level
never above 5.3 m. Standard flow and the wide wall gave the same answers as
now.

### Terrain and meshes at the thin wall

Zoom on the thin wall (black outline) at the centre line. Left: what
standard flow sees, the cell-mean elevations. Right: the SGS mesh, coloured
by each cell's lowest ground, with the faces whose sill the connectivity
search raised drawn in red (`sgs_face_raise`, read from the mesh file with
`FA.load_mesh_columns`). White lines are the cells.

In [ ]:
pw_dir = FC.case_dir(pw)
pwp = Dict(w => FC.planar_params(joinpath(pw_dir, w)) for w in WALLS)
pwt = pwp["thin"]
pw_m_lon = 111320.0 * cosd(pwt["centre_lat"])
pw_lonlat(x, y) = (pwt["lon0"] + x / pw_m_lon, pwt["lat0"] + y / 111320.0)
mcw = FP.MapCoords(:local, pw_lonlat(pwt["emb_x_m"], pwt["notch_y_centre_m"])...)   # metres from the wall's centre line

"The side shared by two cell polygons (two common vertices), projected; nothing if none."
function shared_side(mc, Pi, Pj)
    vi = [(Pi[1, k], Pi[2, k]) for k in 1:size(Pi, 2)]; vj = [(Pj[1, k], Pj[2, k]) for k in 1:size(Pj, 2)]
    c = unique([v for v in vi if any(w -> abs(w[1] - v[1]) < 1e-9 && abs(w[2] - v[2]) < 1e-9, vj)])
    length(c) == 2 || return nothing
    return FP.project(mc, [c[1][1], c[2][1]], [c[1][2], c[2][2]])
end

zrng = (4.6, 5.9)                     # the wall's crest (8.05 m) is far above: the colours resolve the plane
xl = (-70.0, 70.0); yl = (-60.0, 60.0)
fig = Figure()
for (row, mesh) in enumerate(("a5", "cart"))
    stdm = FA.load_mesh_parquet(joinpath(pw_dir, "thin", "$(mesh)_standard.parquet"))
    sgsf = joinpath(pw_dir, "thin", "$(mesh)_sgs.parquet")
    sgm = FA.load_mesh_parquet(sgsf)
    raise = FA.load_mesh_columns(sgsf, ["sgs_face_raise"])["sgs_face_raise"]      # per cell, per neighbour slot
    zmin = FA.MeshGeom(sgm.ids, sgm.lon, sgm.lat, sgm.static["sgs_z_min"], sgm.area, sgm.polygons, sgm.neighbours,
                       sgm.static, sgm.id_scheme)
    for (col, (g, title, sub)) in enumerate(((stdm, "$(PW_MESH[mesh]), standard", "cell-mean elevation"),
                                             (zmin, "$(PW_MESH[mesh]), SGS", "lowest ground per cell; red: faces closed by connectivity")))
        p = FP.fieldmap!(fig[row, col], mesh_run(g, title), 1; var = :elevation, coords = mcw, colorrange = zrng,
                         colormap = :viridis, colorbar = false, sources = false, title, subtitle = sub)
        limits!(p.axis, xl..., yl...)
        p.axis.aspect = nothing; p.axis.width = 420; p.axis.height = 420 * (yl[2] - yl[1]) / (xl[2] - xl[1])
        for P in FP.project_polygons(mcw, g)
            lines!(p.axis, P[1, :], P[2, :]; color = (:white, 0.7), linewidth = 0.6)
        end
        poly!(p.axis, Rect(-2.0, yl[1] - 10, 4.0, yl[2] - yl[1] + 20); color = (:black, 0.0), strokecolor = :black, strokewidth = 1.2)
        col == 2 || continue
        for i in eachindex(g.ids), s in axes(g.neighbours, 1)
            j = g.neighbours[s, i]
            (j == 0 || j < i || s > length(raise[i]) || raise[i][s] <= 0.05) && continue
            seg = shared_side(mcw, g.polygons[i], g.polygons[j]); seg === nothing && continue
            lines!(p.axis, seg[1], seg[2]; color = raise[i][s], colormap = :Reds, colorrange = (0, 3), linewidth = 3)
        end
        r = filter(>(0.05), reduce(vcat, raise))
        @printf("%-15s SGS mesh: %d faces with the sill raised by more than 5 cm, by %.2f–%.2f m\n",
                PW_MESH[mesh], length(r) ÷ 2, extrema(r)...)
    end
end
Colorbar(fig[1:2, 3]; colormap = :viridis, limits = zrng, label = "Elevation (m)", height = Relative(0.7), width = 12)
Colorbar(fig[1:2, 4]; colormap = :Reds, limits = (0, 3), label = "Sill raised by (m)", height = Relative(0.7), width = 12)
resize_to_layout!(fig)
fig

**What standard flow sees.** A cell-mean elevation spreads the 3 m wall over
the whole cell it crosses: a 4 m wall in a 20–22 m cell adds about
3 m × 4/20 ≈ 0.5 m to that cell's mean, less where the wall clips a cell. The
wall becomes a ridge a few decimetres above the plane, with its crest at
the *lowest* such cell along the wall (the lowest cell-mean route across,
in the table above: about 5.4 m on A5 and 5.6 m on the Cartesian grid,
0.3–0.6 m above the ground at the wall, against a crest of 8.05 m). Water
crosses there as soon as the pond reaches it. That is the known limit of
cell-mean beds — the reason SGS exists — and the case checks standard flow
on the thin wall against that ridge, not against the crest.

**What SGS sees.** Its storage tables still treat each cell as one level
pool; the change is in the faces. The connectivity search (HYDRAULICS §9.3)
finds each cell's main basin — on this slope, the low ground on the
downstream side of the wall — and for every point of the cell the level at
which it joins that basin. The points west of the wall join it only over
the crest. A face point conveys only once the water at it is connected to
the main basins of both cells it joins, so the faces on the upstream side
of the wall's cells are closed up to the crest: their sills are raised by
almost the wall's full height (red). The wall is in effect moved onto those
faces — by up to one cell: about 9 m west on the Cartesian grid, where the
closed faces are the west sides of the column holding the wall, and by a
zig-zag of faces on A5.

### Water downstream of the wall, and the level upstream

The same quantities as the case computes, at every output frame (every
2 minutes): the volume in cells entirely east of the wall, and the highest
water level among the wet cells along the wall's upstream foot (the cell
below checks them against the case's hourly values).

In [ ]:
"Per frame: volume east of the wall and the upstream level at its foot — as the case's analysis."
function pw_series(r, p)
    xmin, xmax = FC.embankment_xrange(r.mesh, p)
    xw = p["emb_x_m"] - p["emb_width_m"] / 2; xe = p["emb_x_m"] + p["emb_width_m"] / 2
    cell = sqrt(sum(r.mesh.area) / length(r.mesh.area))
    foot = findall((xmax .<= xw) .& (xmax .>= xw - cell)); down = findall(xmin .>= xe)
    n = FA.nframes(r); wse_up = fill(NaN, n); v_down = zeros(n)
    for k in 1:n
        h = FA.depth(r, k); η = FA.wse(r, k); V = FA.volume(r, k)
        w = [i for i in foot if isfinite(h[i]) && h[i] > 0.01]
        isempty(w) || (wse_up[k] = maximum(η[w]))
        v_down[k] = sum(V[i] for i in down if isfinite(V[i]); init = 0.0)
    end
    return (; t = r.t, wse_up, v_down)
end
pw_runs = Dict(j.name => FA.load_run(FC.job_output(pw, j); label = pw_label(j.name)) for j in pw.jobs)
pw_s = Dict(j => pw_series(r, pwp[String(last(split(j, '_')))]) for (j, r) in pw_runs)
for j in keys(pw_runs), h in 1:6
    k = FA.frame_at(pw_runs[j], 3600.0 * h; tol = 90.0)
    @assert isequal(pw_s[j].wse_up[k], pwm[j]["wse_up_$(h)h"]) || pw_s[j].wse_up[k] ≈ pwm[j]["wse_up_$(h)h"]
    @assert isapprox(pw_s[j].v_down[k], pwm[j]["v_down_$(h)h"]; atol = 1e-6)
end
println("series match the case's hourly metrics for all ", length(pw_runs), " runs")

pw_style(j) = (color = startswith(j, "a5") ? FP.SERIES[1] : FP.SERIES[2],
               linestyle = occursin("_sgs_", j) ? :solid : :dash)
fig = Figure(size = (1150, 760))
for (c, w) in enumerate(WALLS)
    ax1 = Axis(fig[1, c]; xlabel = "Time (h)", ylabel = "Downstream volume (10³ m³)",
               title = "$w wall ($(round(Int, pwp[w]["emb_width_m"])) m): water east of the wall")
    ax2 = Axis(fig[2, c]; xlabel = "Time (h)", ylabel = "Upstream WSE (m)",
               title = "$w wall: water level at its upstream foot")
    for (z, lab) in [(pwm["design_$w"]["crest_z"], "crest"), (pwm["design_$w"]["wse_end"], "level-pool level of the whole inflow"),
                     (pwm["design_$w"]["z_foot"], "ground at the wall")]
        hlines!(ax2, [z]; color = FP.MUTED, linestyle = :dot, linewidth = 1)
        startswith(lab, "ground") ?
            text!(ax2, 5.9, z; text = lab, color = FP.INK2, fontsize = 11, align = (:right, :top)) :
            text!(ax2, 0.1, z; text = lab, color = FP.INK2, fontsize = 11, align = (:left, :bottom))
    end
    for mesh in ("a5", "cart"), model in ("sgs", "std")
        j = pw_job(mesh, model, w); s = pw_s[j]; st = pw_style(j)
        lines!(ax1, s.t ./ 3600, s.v_down ./ 1e3; st..., linewidth = 2, label = pw_label(j))
        lines!(ax2, s.t ./ 3600, s.wse_up; st..., linewidth = 2)
    end
    ylims!(ax1, -5, 200); ylims!(ax2, 4.9, 8.3)
    c == 1 && Legend(fig[3, 1:2], ax1; orientation = :horizontal, framevisible = false)
end
fig

**End-of-run maps, thin wall.** Mean depth V/A at 6 h (cells with less than
1 cm not drawn), SGS against standard on both meshes. The dashed lines mark
the wall; the star is the inflow point. (Mean depth, because an SGS depth is
measured from the cell's lowest ground and a standard depth from its mean.)

In [ ]:
mcx = FP.MapCoords(:local, pw_lonlat(0.0, pwt["notch_y_centre_m"])...)       # x from the west edge, y from the centre line
xw4 = FP.project(mcx, [pw_lonlat(x, 0.0)[1] for x in pwt["emb_x_m"] .+ (-2.0, 2.0)], [pwt["lat0"], pwt["lat0"]])[1]
thin = [pw_job(mesh, model, "thin") for mesh in ("a5", "cart"), model in ("sgs", "std")]
mdw = [meandepth_run(pw_runs[j], FA.nframes(pw_runs[j])) for j in thin]
rng = FP.default_range(:depth, filter(isfinite, reduce(vcat, [x.mesh.elevation for x in vec(mdw)])))
fig = Figure()
for i in 1:2, j in 1:2
    r = pw_runs[thin[i, j]]
    p = FP.fieldmap!(fig[i, j], mdw[i, j], 1; var = :elevation, colormap = cgrad(FP.SEQUENTIAL), basemap = :grey,
                     coords = mcx, colorrange = rng, colorbar = false, sources = false,
                     title = r.label, subtitle = FP.time_label(r.t[end]) * " · mean depth", width = 520)
    vlines!(p.axis, collect(xw4); color = FP.INK2, linestyle = :dash, linewidth = 0.8)
    FP.mark_sources!(p, r; labels = false)
    i < 2 && hidexdecorations!(p.axis; grid = false, ticks = false)
    j > 1 && hideydecorations!(p.axis; grid = false, ticks = false)
end
Colorbar(fig[1:2, 3]; colormap = cgrad(FP.SEQUENTIAL), limits = rng, label = "Mean depth V/A (m)", height = Relative(0.6), width = 12)
resize_to_layout!(fig)
fig

### Interpretation

- **SGS now holds both walls, on both meshes.** No water reaches the
  downstream side in any SGS run. Against the thin wall the pond rises to
  within a few centimetres of the level-pool level of the whole inflow
  (about 7.1 m), 0.9 m below the crest — where under the 0.6.0 scheme it
  never rose above 5.3 m because 58–64% of the inflow had passed the wall,
  from the moment the water reached it.
- **Why.** The level-pool storage is unchanged; what changed is that a face
  conveys only from the level at which its ground is connected, below the
  water, to the main water bodies of the two cells it joins. For the cells
  the thin wall crosses, the main basin lies on the wall's downstream side,
  so the faces on its upstream side open only at the crest. The hop is
  closed by the tables alone, with no cost in the time step.
- **Standard flow overtops the thin wall early, as expected.** Its
  cell-mean beds turn the 3 m wall into a ridge of a few decimetres, and
  water crosses as soon as the pond reaches that ridge (after about 45 min on
  A5 and an hour on the Cartesian grid); by 6 h more than 80% of the inflow
  is downstream. This is not what the case is for, but it is the reason a
  sub-grid model is wanted at all.
- **The wide wall holds in every run**, SGS and standard: some cells lie
  wholly on its top, so every route across passes a face on the wall. (Under
  the 0.6.0 scheme SGS reached only about 6.6 m at the wide wall because its
  conveyance defects held water near the source; now its pond reaches the
  level-pool level like standard flow's.)
- **The limit.** One storage per cell: the water of a cell is one pool at one
  level. The wall's effect is therefore placed on the faces of the cells it
  crosses — the barrier is moved by up to one cell, and the water stored in
  the cut-off part of a wall cell is counted at the level of the pool it is
  separated from. A cell holding two water bodies of comparable importance
  (a river and a side channel behind a levee) is the case this cannot
  represent; splitting such cells into one storage per basin is the next
  step (HYDRAULICS §9.3, "Limit").

## 5. Summary and where to go next

| Case | Result |
|---|---|
| `planar_slope` | Mass conserved to round-off. Cartesian 25 m symmetric (asym ≈ 10⁻⁵). A5 res 16 and 18: asym ≈ +0.09 and deflection ≤ 3° at 5–20 h (0.68–0.87 and 14–15° before the 0.3.0 scheme); flow downslope. |
| `planar_embankment` | SGS passes water through a 4 m notch (a fifth of a cell) after about 14 minutes at 5.25 m upstream, and nowhere else; the notch then carries nearly all the inflow. Standard holds it until 4.5 h and 6.8 m (its cell-mean crest 6.72 m). Mass conserved. |
| `planar_wall` | No water passes a wall without a gap in any SGS run, 40 m or 4 m wide (0.6.0: 58–64% passed the 4 m wall); the pond reaches its level-pool level. Standard flow holds the 40 m wall and overtops the 4 m one at its cell-mean ridge, as cell-mean beds must. |

**Reproduce one run by hand.** Copy a `run …` command from `FC.show_case`
above into a terminal at the repository root (prefix it with
`FLOODA5_NO_VIS=1`). Every output file also records the command that made
it, and `--rerun` repeats it exactly from the file alone:

```bash
julia --project=. --threads 4 FloodModel.jl --rerun test/case_runs/planar_embankment/runs/sgs.h5 --output sgs_again.h5
```

The command line recorded in a file: `FA.load_run(path).info["argv"]`.

**Run the cases from a terminal** (`cases/run.jl`, the same registry):

```bash
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl show planar_embankment     # the commands
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl run planar_slope planar_embankment planar_wall -P 2 -T 3
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl check planar_slope        # analyse existing runs, check
```

**Change a case.** Edit `cases/registry/planar_slope.jl`,
`planar_embankment.jl` or `planar_wall.jl` (e.g. the notch width, the inflow, the meshes). Runs
whose command changed are made again; others are reused. After a change to
the model code, add `--fresh` (`FC.run_case(c; fresh = true)`) to re-run
everything. If the results change on purpose, record the new expected values
with `cases/run.jl baseline CASE` (it writes `cases/expected/CASE.toml` with
the version and commit) and say why in `CHANGELOG.md`.

**Watch a run.** Replay an output in the interactive GLMakie viewer
(needs the full project and a display, so not from this notebook's kernel):

```bash
julia --project=. --threads auto visualisation/replay.jl test/case_runs/planar_embankment/runs/sgs.h5
```

**Further reading.** `docs/TEST_CASES.md` §2 (§2a, 2b, 2d: these cases),
`docs/HYDRAULICS.md` §7–8 and §15 (directional bias and the schemes that
removed it), §9 (the SGS solver: tables, faces, connectivity),
`CHANGELOG.md` (the 0.7.0 SGS rewrite), `notebooks/04_sgs_strip.ipynb` (SGS
against analytic answers), `test/planar_embankment/README.md` (the
terrain generator), `docs/USER_GUIDE.md` §6.5 (the figure functions used here).